# Phase-1 KDE: ship vs A2 vs A3 α=1 (gap-only selectors)

**Intent:** compare three training-set selectors at T-5d, T-4d, T-3d, T-2d, T-1d.

1. **Ship baseline:** A3 (combined_score, α=0.5, σ_gap=8) + C2 weighted KDE + E2 weighted base_rate.
2. **A2 variant:** `matched_training_slugs(band=3d, n=20)` — hard gap filter (expands band until 20 found), recency tiebreak. **Note:** A2 produces no per-movie weights, so C2/E2 degrade to **C1/E1** (unweighted KDE, unweighted base_rate). This is an unavoidable consequence of the selector shape, not an intentional change.
3. **A3 α=1 variant:** A3 combined_score with α=1.0 — pure gap-score weighting, no Jaccard. Still uses C2+E2 (weighted KDE/base_rate, weights = gap_scores).

**Knobs held fixed across variants:** B1 (n=20), D2+D3 (bw 0.5–0.7d), F1 (scaling thr=40 clamp (0.5, 2.0)), G2 (midnight snap), H2 (noon-shift), I2 (phase-1 to midnight UTC), K1 (no boost). Phase 2 not evaluated.

**Decision lens:** per-target paired bootstrap CI on (ship − A2) and (ship − A3_α1).


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import time

import numpy as np
import pandas as pd

import _helpers as H

print(f'cohort: {len(H.close_date_map)} resolved movies  ·  reviews: {len(H.reviews)}')


## Apply noon-shift (H2) + recompute derived artifacts


In [ ]:
_day_mask = H.reviews['timestamp_confidence'] == 'd'
_n_shifted = int(_day_mask.sum())
H.reviews.loc[_day_mask, 'estimated_timestamp'] = (
    H.reviews.loc[_day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)

H.first_review_ts = (
    H.reviews[H.reviews['movie_slug'].isin(H.close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
_new_first = H.first_review_ts.to_dict()
H.gaps['first_review_ts'] = H.gaps['slug'].map(_new_first)
H.gaps['gap_days'] = (
    H.gaps['close_ts'] - H.gaps['first_review_ts']
).dt.total_seconds() / 86400
H.gap_lookup = dict(zip(H.gaps['slug'], H.gaps['gap_days']))

print(f'noon-shift: moved {_n_shifted} day-level reviews (+12h)')
print(f'gap_days now  median={H.gaps["gap_days"].median():.2f}  '
      f'IQR={H.gaps["gap_days"].quantile(0.75) - H.gaps["gap_days"].quantile(0.25):.2f}')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
SHIP_ALPHA = 0.5
SIGMA_GAP = 8.0
N_TRAINING = 20
A2_BAND = 3.0
BANDWIDTH_FLOOR = 0.5
BANDWIDTH_CEILING = 0.7
SHRINKAGE_K = 3.0
SCALING_THRESHOLD = 40.0
SCALING_CLAMP = (0.5, 2.0)
MIN_TRAINING_SCORES = 5
CHECKPOINT_EVERY = 300

CACHE_PATH = H.CACHE_DIR / 'phase1_a2_vs_a3alpha1.pkl'
VARIANTS = ['ship', 'A2', 'A3_alpha1']
print(f'cache: {CACHE_PATH}')


## Per-(target, snap, variant) evaluation

Shared skip-rule gating across all three variants (so sample sets are comparable). A2's C1+E1 downgrade is handled by using `build_critic_profiles` + `build_kde_lambda_model_capped` (the unweighted KDE path) when variant == 'A2'.


In [ ]:
_error_counts = {v: 0 for v in VARIANTS}
_error_samples = []


def eval_target_snap(target_slug, snap_days, variant):
    try:
        close_ts = H.close_date_map[target_slug]
        midnight_utc = close_ts.floor('D')
        midnight_utc_dbc = (close_ts - midnight_utc).total_seconds() / 86400

        snap_time = midnight_utc - pd.Timedelta(days=snap_days)
        snap_dbc_effective = (close_ts - snap_time).total_seconds() / 86400

        state = H.snapshot_state(target_slug, snap_time)
        ok, _ = H.passes_skip_rules_for_snap(state, snap_dbc_effective)
        if not ok:
            return None

        target_gap = H.gap_lookup.get(target_slug)
        if target_gap is None:
            return None

        target_window_days = state['first_review_dbc'] - snap_dbc_effective
        if target_window_days <= 0:
            return None

        if variant == 'ship':
            scores = H.combined_score_with_scores(
                target=target_slug, target_gap=target_gap,
                target_critics=state['observed_critics'],
                target_window_days=target_window_days,
                k=N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SIGMA_GAP,
            )
            if len(scores) < MIN_TRAINING_SCORES:
                return None
            profiles = H.build_weighted_critic_profiles(
                H.reviews, H.close_date_map, scores,
            )
            model = H.build_weighted_kde_lambda_model(
                profiles, shrinkage_k=SHRINKAGE_K,
                bandwidth_floor=BANDWIDTH_FLOOR, bandwidth_ceiling=BANDWIDTH_CEILING,
            )
        elif variant == 'A3_alpha1':
            scores = H.combined_score_with_scores(
                target=target_slug, target_gap=target_gap,
                target_critics=state['observed_critics'],
                target_window_days=target_window_days,
                k=N_TRAINING, alpha=1.0, sigma_gap=SIGMA_GAP,
            )
            if len(scores) < MIN_TRAINING_SCORES:
                return None
            profiles = H.build_weighted_critic_profiles(
                H.reviews, H.close_date_map, scores,
            )
            model = H.build_weighted_kde_lambda_model(
                profiles, shrinkage_k=SHRINKAGE_K,
                bandwidth_floor=BANDWIDTH_FLOOR, bandwidth_ceiling=BANDWIDTH_CEILING,
            )
        elif variant == 'A2':
            slugs, _ = H.matched_training_slugs(
                target_slug, target_gap, band=A2_BAND, n=N_TRAINING,
            )
            if len(slugs) < MIN_TRAINING_SCORES:
                return None
            profiles = H.build_critic_profiles(
                H.reviews, H.close_date_map, slugs, verbose=False,
            )
            model = H.build_kde_lambda_model_capped(
                profiles, shrinkage_k=SHRINKAGE_K,
                bandwidth_floor=BANDWIDTH_FLOOR, bandwidth_ceiling=BANDWIDTH_CEILING,
            )
        else:
            raise ValueError(variant)

        pred = H.predict_window_custom(
            model=model,
            dbc_from=snap_dbc_effective,
            dbc_to=midnight_utc_dbc,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
            scaling_threshold=SCALING_THRESHOLD,
            scaling_clamp=SCALING_CLAMP,
        )

        if not np.isfinite(pred):
            _error_counts[variant] += 1
            return None

        actual = H.actual_in_window(target_slug, snap_dbc_effective, midnight_utc_dbc)

        return {
            'target_slug': target_slug,
            'snap_days': snap_days,
            'variant': variant,
            'pred': float(pred),
            'actual': int(actual),
            'err': float(pred) - int(actual),
            'observed_count': state['observed_count'],
            'first_review_dbc': state['first_review_dbc'],
            'target_gap': float(target_gap),
        }
    except Exception as exc:
        _error_counts[variant] = _error_counts.get(variant, 0) + 1
        if len(_error_samples) < 5:
            _error_samples.append({
                'target': target_slug, 'snap': snap_days, 'variant': variant,
                'reason': f'{type(exc).__name__}: {exc}',
            })
        return None


## Smoke-test


In [ ]:
_smoke_slugs = [s for s in H.close_date_map if s not in {'the_drama', 'the_super_mario_galaxy_movie'}]
_smoke_target = sorted(_smoke_slugs, key=lambda s: H.close_date_map[s], reverse=True)[0]
print(f'smoke target: {_smoke_target}   close={H.close_date_map[_smoke_target]}')
for snap_days in [5, 3, 1]:
    rs = {v: eval_target_snap(_smoke_target, snap_days, v) for v in VARIANTS}
    if all(r is None for r in rs.values()):
        print(f'  T-{snap_days}d: all skipped')
        continue
    parts = []
    for v, r in rs.items():
        if r is None:
            parts.append(f'{v}=skip')
        else:
            parts.append(f'{v}_pred={r["pred"]:.2f} err={r["err"]:+.2f}')
    actual = next((r['actual'] for r in rs.values() if r is not None), '?')
    print(f'  T-{snap_days}d  actual={actual}  ' + '  '.join(parts))


## LOO sweep


In [ ]:
def run_loo(force=False):
    if CACHE_PATH.exists() and not force:
        with open(CACHE_PATH, 'rb') as f:
            cached = pickle.load(f)
        print(f'loaded cached {len(cached)} rows')
        return cached

    all_targets = sorted(H.close_date_map.keys())
    results = []
    start = time.time()
    total_combos = len(all_targets) * len(SNAP_DAYS_LIST) * len(VARIANTS)
    done = 0

    for target_slug in all_targets:
        for snap_days in SNAP_DAYS_LIST:
            for variant in VARIANTS:
                r = eval_target_snap(target_slug, snap_days, variant)
                if r is not None:
                    results.append(r)
                done += 1

            if done % CHECKPOINT_EVERY == 0:
                elapsed = time.time() - start
                eta = elapsed / done * (total_combos - done)
                err_counts = ', '.join(f'{k}={v}' for k, v in _error_counts.items())
                print(f'  {done}/{total_combos}  kept {len(results)}  '
                      f'err({err_counts})  '
                      f'elapsed {elapsed/60:.1f}m  eta {eta/60:.1f}m')
                with open(CACHE_PATH, 'wb') as f:
                    pickle.dump(pd.DataFrame(results), f)

    df = pd.DataFrame(results)
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(df, f)
    print(f'\nsaved {len(df)} rows to {CACHE_PATH}')
    if _error_samples:
        print(f'{sum(_error_counts.values())} errored combos. Samples:')
        for s in _error_samples:
            print(f'  {s}')
    return df

df = run_loo()
print(f'total rows: {len(df)}')
df.head()


## Per-variant, per-snap summary


In [ ]:
def row_metrics(sub):
    err = sub['err'].values
    abs_err = np.abs(err)
    pred = sub['pred'].values
    actual = sub['actual'].values.astype(float)
    safe_actual = np.where(actual > 0, actual, np.nan)
    return {
        'n': len(sub),
        'MAE': float(abs_err.mean()) if len(sub) else np.nan,
        'me': float(err.mean()) if len(sub) else np.nan,
        'med_err': float(np.median(err)) if len(sub) else np.nan,
        'med_abs_err': float(np.median(abs_err)) if len(sub) else np.nan,
        'p90_abs_err': float(np.quantile(abs_err, 0.9)) if len(sub) else np.nan,
        'med_ratio': float(np.nanmedian(pred / safe_actual)) if len(sub) else np.nan,
    }


rows = []
for snap_days in SNAP_DAYS_LIST:
    for variant in VARIANTS:
        sub = df[(df['variant'] == variant) & (df['snap_days'] == snap_days)]
        m = row_metrics(sub)
        rows.append({'snap_days': snap_days, 'variant': variant, **m})
summary = pd.DataFrame(rows)

print('=== per-variant summary ===\n')
for snap_days in SNAP_DAYS_LIST:
    print(f'T-{snap_days}d')
    for variant in VARIANTS:
        m = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == variant)].iloc[0]
        print(f'  {variant:12s}  n={int(m["n"]):3d}  MAE={m["MAE"]:6.2f}  '
              f'me={m["me"]:+6.2f}  med_err={m["med_err"]:+6.2f}  '
              f'med|e|={m["med_abs_err"]:6.2f}  p90|e|={m["p90_abs_err"]:6.2f}  '
              f'med_ratio={m["med_ratio"]:5.2f}')
    print()


## Side-by-side Δ vs ship


In [ ]:
print(f'{"snap":<6}{"variant":<14}{"MAE":>8}{"Δ units":>10}{"Δ %":>9}{"me":>8}{"ship me":>10}')
for snap_days in SNAP_DAYS_LIST:
    ship_row = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == 'ship')].iloc[0]
    for variant in VARIANTS:
        m = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == variant)].iloc[0]
        if variant == 'ship':
            print(f'T-{snap_days}d  {variant:<14}{m["MAE"]:>8.2f}{"—":>10}{"—":>9}'
                  f'{m["me"]:>+8.2f}{ship_row["me"]:>+10.2f}')
        else:
            delta = ship_row['MAE'] - m['MAE']
            pct = 100 * delta / ship_row['MAE']
            print(f'T-{snap_days}d  {variant:<14}{m["MAE"]:>8.2f}{delta:>+10.2f}{pct:>+8.2f}%'
                  f'{m["me"]:>+8.2f}{ship_row["me"]:>+10.2f}')
    print()


## Paired bootstrap CI (ship minus each variant)

Per target: delta = |err_ship| − |err_variant|. Positive = ship wins.


In [ ]:
print(f'{"snap":<6}{"variant":<14}{"Δ units":>10}{"CI95_lo":>10}{"CI95_hi":>10}'
      f'{"Δ %":>10}{"n":>6}  sig')

ci_rows = []
for snap_days in SNAP_DAYS_LIST:
    ship_sub = df[(df['variant'] == 'ship') & (df['snap_days'] == snap_days)]
    for variant in VARIANTS:
        if variant == 'ship':
            continue
        v_sub = df[(df['variant'] == variant) & (df['snap_days'] == snap_days)]
        merged = ship_sub.merge(v_sub, on='target_slug', suffixes=('_ship', '_v'))
        if len(merged) == 0:
            continue
        ship_abs = np.abs(merged['err_ship'].values)
        v_abs = np.abs(merged['err_v'].values)
        # delta > 0 → variant has smaller MAE than ship → variant better
        # (matches H.bootstrap_mae_delta convention where +ve = 'method' wins over control)
        deltas = ship_abs - v_abs
        point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
        ship_mae = ship_abs.mean()
        pct = 100 * point / ship_mae
        sig = 'SIG' if lo > 0 else ('sig-neg' if hi < 0 else 'ns')
        ci_rows.append({
            'snap_days': snap_days, 'variant': variant,
            'delta': point, 'ci_lo': lo, 'ci_hi': hi,
            'delta_pct': pct, 'n_paired': len(merged), 'sig': sig,
        })
        print(f'T-{snap_days}d  {variant:<14}{point:>+10.3f}{lo:>+10.3f}{hi:>+10.3f}'
              f'{pct:>+10.2f}{len(merged):>6}  {sig}')
    print()
ci_df = pd.DataFrame(ci_rows)


## Plot: MAE by snap by variant


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
colors = {'ship': 'tab:red', 'A2': 'tab:orange', 'A3_alpha1': 'tab:blue'}
for variant in VARIANTS:
    sub = summary[summary['variant'] == variant].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['MAE'], 'o-', label=variant, color=colors.get(variant))
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('MAE')
ax.set_title('Phase-1 MAE by snap and selector')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Plot: mean_error by snap by variant


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
for variant in VARIANTS:
    sub = summary[summary['variant'] == variant].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['me'], 'o-', label=variant, color=colors.get(variant))
ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('mean error (pred − actual)')
ax.set_title('Phase-1 bias by snap and selector')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
